### 3.4.14. Solver Interfaces for Convex Models

$$
\begin{aligned}
\min_x\quad & f_0(x) \\
\text{subject to}\quad & f_i(x)\leq 0,\quad i=1,\ldots,m,\\
& h_j(x)=0,\quad j=1,\ldots,p
\end{aligned}
\qquad \longrightarrow \qquad
\text{solver data}
$$


**Explanation:**

A solver interface records the objective, decision variables, constraints, and structural class of a convex model, then passes the corresponding data to a numerical solver. The interface layer is not the hierarchy itself; the detailed class-by-class organization appears in [Convex Programming Problem Classes](../03_Convex_Optimization/30_convex_programming_problem_classes.ipynb). Here the optimization-specific point is practical: once a model is recognized as an LP, QP, SOCP, SDP, or cone program, the narrowest correct solver class is usually the cleanest computational target.

The display shows the mathematical model on the left and the solver-ready representation on the right. The same feasible set and objective should be preserved, even though the data format changes from human notation to matrices, cone memberships, and bounds.

**Properties:**
- The modeling layer separates the mathematical formulation from a solver backend.
- Structural recognition should preserve objective, feasibility, and optimality.
- A broader solver can often represent a narrower class, but specialized solvers can exploit extra structure.

**Intuition:**

The calculation below stores a small LP in matrix form and also notes that it can be embedded in a QP template by choosing zero quadratic curvature. The point is not to re-teach the class hierarchy, but to show why the interface should still report the narrowest useful class: this model is an LP even though a QP solver could accept it.


**Numerical Example:**

Consider the linear program
$$
\min_{x_1,x_2}\ x_1+2x_2
\quad\text{subject to}\quad
x_1+x_2=1,
\quad x_1\geq0,
\quad x_2\geq0.
$$

A solver interface can store this as
$$
c=\begin{bmatrix}1\\2\end{bmatrix},\qquad
A_{\rm eq}=\begin{bmatrix}1&1\end{bmatrix},\qquad
b_{\rm eq}=1,
$$
with nonnegativity bounds on $x$. If the same interface sends the model to a QP-capable backend, it can set
$$
P=\begin{bmatrix}0&0\\0&0\end{bmatrix},
\qquad
\frac12x^TPx+c^Tx=c^Tx.
$$

The feasible vertices are $(1,0)$ and $(0,1)$. Their objective values are
$$
c^T\begin{bmatrix}1\\0\end{bmatrix}=1,
\qquad
c^T\begin{bmatrix}0\\1\end{bmatrix}=2.
$$
Thus the optimizer is $x^\star=(1,0)$ and the minimum value is $1$. The zero eigenvalues of $P$ show that the QP embedding is convex, but the narrowest class remains LP.


In [1]:
import sympy as sp

cost = sp.Matrix([1, 2])
equality_matrix = sp.Matrix([[1, 1]])
equality_value = sp.Matrix([1])
quadratic_matrix = sp.zeros(2)
vertices = [sp.Matrix([1, 0]), sp.Matrix([0, 1])]
objective_values = [(cost.T * vertex)[0] for vertex in vertices]
best_index = min(range(len(vertices)), key=lambda index: objective_values[index])

print("c ="); sp.pprint(cost)
print("A_eq =", equality_matrix)
print("b_eq =", equality_value)
print("P eigenvalues =", quadratic_matrix.eigenvals())
print("narrowest_class = LP")
print("optimizer ="); sp.pprint(vertices[best_index])
print("minimum_value =", objective_values[best_index])


c =
⎡1⎤
⎢ ⎥
⎣2⎦
A_eq = Matrix([[1, 1]])
b_eq = Matrix([[1]])
P eigenvalues = {0: 2}
narrowest_class = LP
optimizer =
⎡1⎤
⎢ ⎥
⎣0⎦
minimum_value = 1


**Equivalent `casadi` implementation:**


In [2]:
import casadi as ca

opti = ca.Opti()
decision_vector = opti.variable(2)
linear_cost = ca.DM([1, 2])
objective = ca.dot(linear_cost, decision_vector)

opti.minimize(objective)
opti.subject_to(decision_vector[0] + decision_vector[1] == 1)
opti.subject_to(decision_vector >= 0)
opti.set_initial(decision_vector, ca.DM([0.5, 0.5]))
opti.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes"})
solution = opti.solve()

print("narrowest_class = LP")
print("optimizer =", ca.DM(solution.value(decision_vector)))
print("minimum_value =", ca.DM(solution.value(opti.f)))


narrowest_class = LP
optimizer = [1, -7.4907e-09]
minimum_value = 1


**References:**

[📗 Tordesillas, J. (n.d.). *Optimization, Optimal Control, Trajectory Optimization, and Splines*.](https://www.youtube.com/watch?v=j82Ia436DYY)  
[📗 Jasour, A. (2019). *MIT 16.S498 Lecture 2: Nonlinear Optimization Overview*.](https://github.com/jasour/rarnop19/tree/master/Lecture2_NonlinearOptimization)  
[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)

---

[⬅️ Previous: Classification and Support Vector Machines](./13_classification_and_support_vector_machines.ipynb) | [Next: General Nonlinear Programs ➡️](../05_Nonlinear_Programming/01_general_nonlinear_programs.ipynb)

---
